# Model Training and Evaluation
## Student Performance Prediction

This notebook demonstrates the complete machine learning pipeline for predicting student performance.

In [ ]:
# Add src to path
import sys
sys.path.append('../src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from train_model import StudentPerformancePredictor

import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

## 1. Initialize Predictor and Load Data

In [ ]:
# Initialize predictor
predictor = StudentPerformancePredictor('../data/global_university_students_performance_habits_10000.csv')

# Load data
df = predictor.load_data()
df.head()

## 2. Data Overview

In [ ]:
print(f"Dataset shape: {df.shape}")
print(f"\nTarget variable statistics:")
print(df['final_exam_score'].describe())

## 3. Preprocess Data

In [ ]:
# Preprocess for final_exam_score prediction
X_train, X_test, y_train, y_test = predictor.preprocess_data(target='final_exam_score')

print(f"Training features shape: {X_train.shape}")
print(f"Test features shape: {X_test.shape}")
print(f"\nFeature names: {list(X_train.columns)}")

## 4. Train Models

In [ ]:
# Train all models
predictor.train_models()

## 5. Evaluate Models

In [ ]:
# Get evaluation results
results_df, best_model_name = predictor.evaluate_models()

# Display results
print("\nModel Performance Comparison:")
results_df.style.background_gradient(cmap='RdYlGn', subset=['R2'])

## 6. Visualize Results

In [ ]:
# Create visualizations
predictor.plot_results(save_path='../results')

## 7. Feature Importance Analysis

In [ ]:
# Get best model
best_model = predictor.models[best_model_name]

# Feature importance (if available)
if hasattr(best_model, 'feature_importances_'):
    feature_importance = pd.DataFrame({
        'Feature': X_train.columns,
        'Importance': best_model.feature_importances_
    }).sort_values('Importance', ascending=False)
    
    print("\nTop 15 Most Important Features:")
    print(feature_importance.head(15))
    
    # Plot
    plt.figure(figsize=(10, 6))
    plt.barh(feature_importance.head(15)['Feature'], feature_importance.head(15)['Importance'])
    plt.xlabel('Importance')
    plt.title(f'Feature Importance - {best_model_name}')
    plt.gca().invert_yaxis()
    plt.tight_layout()
    plt.show()

## 8. Prediction Examples

In [ ]:
# Make predictions on test set
y_pred = best_model.predict(X_test)

# Compare predictions with actual values
comparison = pd.DataFrame({
    'Actual': y_test.values[:10],
    'Predicted': y_pred[:10],
    'Difference': (y_pred[:10] - y_test.values[:10])
})

print("\nFirst 10 Predictions vs Actual:")
print(comparison)

## 9. Error Analysis

In [ ]:
# Calculate errors
errors = y_pred - y_test.values

# Plot error distribution
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.hist(errors, bins=30, edgecolor='black', alpha=0.7)
plt.xlabel('Prediction Error')
plt.ylabel('Frequency')
plt.title('Distribution of Prediction Errors')
plt.axvline(x=0, color='red', linestyle='--', label='Perfect Prediction')
plt.legend()

plt.subplot(1, 2, 2)
plt.scatter(y_test, errors, alpha=0.5)
plt.xlabel('Actual Score')
plt.ylabel('Prediction Error')
plt.title('Residual Plot')
plt.axhline(y=0, color='red', linestyle='--')
plt.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nError Statistics:")
print(f"Mean Error: {errors.mean():.2f}")
print(f"Std Error: {errors.std():.2f}")
print(f"Max Over-prediction: {errors.max():.2f}")
print(f"Max Under-prediction: {errors.min():.2f}")

## 10. Save Model

In [ ]:
# Save the best model
model_path = predictor.save_best_model(save_path='../models')
print(f"Model saved successfully to: {model_path}")

## Summary

This notebook demonstrated:
1. Loading and preprocessing student performance data
2. Training 8 different regression models
3. Evaluating model performance
4. Analyzing feature importance
5. Making predictions and analyzing errors
6. Saving the best model for deployment

**Next Steps:**
- Use the saved model for predictions on new data
- Implement the model in a web application
- Continue monitoring and improving model performance